# Chapter 7 - Lab 3: <font color='blue'>Conversational Multi-Agent with AutoGen</font>

**<font color='purple'>Goal</font>**:
Solve a financial-analysis task with AutoGen's **conversational style** using the modern v0.4+ API: two agents — an `AssistantAgent` that *writes Python code* and a `CodeExecutorAgent` that *runs it* — exchange messages until the task is complete.

Unlike Labs 1 and 2, there are **no handoffs and no shared state**. The team's round-robin chat is the coordination mechanism. The Assistant generates code that pulls live data via `yfinance`, the CodeExecutorAgent runs it, and they iterate.

Note: this lab uses the same AutoGen v0.4+ API as Chapter 3 Lab 4 (`autogen-agentchat`, `autogen-ext`, `autogen-core`). Older `pyautogen` v0.2 code is not used.

**<font color='purple'>Tech stack</font>**:

* **AutoGen v0.4+** (`autogen-agentchat`, `autogen-ext`, `autogen-core`).
* `LocalCommandLineCodeExecutor` from `autogen_ext.code_executors.local`.
* **yfinance** — live market data (no API key needed).
* **OpenAI** `gpt-4o`.

## 1. Install packages

In [ ]:
%pip install -q autogen-agentchat autogen-ext autogen-core yfinance pandas matplotlib python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    pass

## 3. Create the model client

AutoGen v0.4 separates the model client from the agent. `OpenAIChatCompletionClient` talks to OpenAI's API; you would swap it for a different client to switch providers.

In [ ]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model='gpt-4o',
    api_key=os.environ['OPENAI_API_KEY'],
)

## 4. Create the code executor

`LocalCommandLineCodeExecutor` runs generated Python in a temporary directory. We cap execution at 60 seconds — long enough for charts but not for runaway loops. In v0.4 the executor needs to be started inside an async context manager.

In [ ]:
import tempfile
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor

work_dir = tempfile.mkdtemp()
code_executor = LocalCommandLineCodeExecutor(timeout=60, work_dir=work_dir)
await code_executor.start()

## 5. Define the two agents

The **`AssistantAgent`** is the brains — it writes Python code that fetches data and produces analysis. The **`CodeExecutorAgent`** runs the code blocks the assistant emits. In v0.4 the code-executor agent is a first-class type — no need for `UserProxyAgent` tricks.

In [ ]:
from autogen_agentchat.agents import AssistantAgent, CodeExecutorAgent

assistant = AssistantAgent(
    name='assistant',
    model_client=model_client,
    system_message=(
        'You are a financial analyst assistant. When asked to analyse a company, '
        'write Python code using yfinance to fetch financial data and compute key metrics. '
        'Present results in a clear, structured format. End your final message with TERMINATE.'
    ),
)

executor_agent = CodeExecutorAgent(
    name='code_executor',
    code_executor=code_executor,
)

## 6. Wire them into a team and run

The two agents take turns inside a `RoundRobinGroupChat`. Termination conditions stop the conversation either when the assistant emits `TERMINATE` (`TextMentionTermination`) or after 10 messages (`MaxMessageTermination`), whichever comes first.

In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

termination = TextMentionTermination('TERMINATE') | MaxMessageTermination(10)

team = RoundRobinGroupChat(
    participants=[assistant, executor_agent],
    termination_condition=termination,
)

query = (
    'What is the current price and key financial metrics of NVIDIA? '
    'Plot the stock price over the last 6 months.'
)

result = await team.run(task=query)

## 7. Inspect the conversation

`result.messages` lists every turn. Walk it to see how the assistant's code and the executor's output alternated.

In [ ]:
for msg in result.messages:
    source = getattr(msg, 'source', '?')
    content = getattr(msg, 'content', '')
    print(f'[{source}] {str(content)[:200]}\n')

# Tidy up the code executor's working directory.
await code_executor.stop()

## 8. Results

Three architectures for the same job: pipeline (Lab 1), leader-follower (Lab 2), conversation (Lab 3). **What to notice about conversational agents:**

* **Code generation is the killer feature.** When the assistant can write and execute   code, it can pull arbitrary data, compute anything, and even draw charts. No tool   catalogue needed.
* **Coordination is implicit.** No shared state, no handoffs — just two agents talking.   Simple to set up, harder to debug when something goes wrong mid-conversation.
* **Sandboxing matters.** Generated code runs locally. In production, replace   `LocalCommandLineCodeExecutor` with a sandboxed runtime (Docker, Firecracker, etc).
* **Pick the style that fits the problem.** Sequential for audited workflows (insurance,   KYC). Leader-follower for routing-heavy front desks. Conversational for exploratory analysis.